<p align="right"><a href="./C2_W4_Lab_01_Decision_Trees.ipynb">English</a> | <strong>中文</strong></p>

# Ungraded Lab: 决策树（Decision Trees）

在本 notebook 中，你将可视化决策树如何用信息增益（information gain）进行分裂。

我们会重新使用视频课程里用过的数据集。数据集是：

如你在课程里所见，在决策树中，我们通过考察某次分裂能带来的 **信息增益（information gain）** 来决定一个节点是否分裂。（视频里 IG 的图）

其中

$$\text{Information Gain} = H(p_1^\text{node})- \left(w^{\text{left}}H\left(p_1^\text{left}\right) + w^{\text{right}}H\left(p_1^\text{right}\right)\right),$$

$H$ 是熵（entropy），定义为

$$H(p_1) = -p_1 \text{log}_2(p_1) - (1- p_1) \text{log}_2(1- p_1)$$

记住这里的 log 定义为以 2 为底。运行下面的代码块，亲眼看看熵 $H(p)$ 随 $p$ 变化的行为。

注意 H 在 $p = 0.5$ 时取最大值，这意味着事件发生的概率是 $0.5$；它的最小值在 $p = 0$ 和 $p = 1$ 处取得，即事件是否发生完全可预测。因此，熵表示一个事件的可预测程度。

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from utils import *

In [2]:
%matplotlib widget
_ = plot_entropy()


Canvas(toolbar=Toolbar(toolitems=[('Home', 'Reset original view', 'home', 'home'), ('Back', 'Back to previous …

|                                                     |   Ear Shape | Face Shape | Whiskers |   Cat  |
|:---------------------------------------------------:|:---------:|:-----------:|:---------:|:------:|
| <img src="images/0.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Present  |    1   |
| <img src="images/1.png" alt="drawing" width="50"/> |   Floppy   |  Not Round  |  Present  |    1   |
| <img src="images/2.png" alt="drawing" width="50"/> |   Floppy   |  Round      |  Absent   |    0   |
| <img src="images/3.png" alt="drawing" width="50"/> |   Pointy   |  Not Round  |  Present  |    0   |
| <img src="images/4.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Present  |    1   |
| <img src="images/5.png" alt="drawing" width="50"/> |   Pointy   |   Round     |  Absent   |    1   |
| <img src="images/6.png" alt="drawing" width="50"/> |   Floppy   |  Not Round  |  Absent   |    0   |
| <img src="images/7.png" alt="drawing" width="50"/> |   Pointy   |  Round      |  Absent   |    1   |
| <img src="images/8.png" alt="drawing" width="50"/> |    Floppy  |   Round     |  Absent   |    0   |
| <img src="images/9.png" alt="drawing" width="50"/> |   Floppy   |  Round      |  Absent   |    0   |


我们会用 **one-hot 编码** 来编码这些类别型特征，如下：

- Ear Shape（耳形）：Pointy = 1, Floppy = 0
- Face Shape（脸形）：Round = 1, Not Round = 0
- Whiskers（胡须）：Present = 1, Absent = 0

因此，我们有两个集合：

- `X_train`：对每个样本，含 3 个特征：
            - Ear Shape（尖耳为 1，否则为 0）
            - Face Shape（圆脸为 1，否则为 0）
            - Whiskers（有胡须为 1，否则为 0）

- `y_train`：该动物是否为猫
            - 是猫为 1
            - 否则为 0

In [3]:
X_train = np.array([[1, 1, 1],
[0, 0, 1],
 [0, 1, 0],
 [1, 0, 1],
 [1, 1, 1],
 [1, 1, 0],
 [0, 0, 0],
 [1, 1, 0],
 [0, 1, 0],
 [0, 1, 0]])

y_train = np.array([1, 1, 0, 0, 1, 1, 0, 1, 0, 0])

In [4]:
#For instance, the first example
X_train[0]

array([1, 1, 1])

这表示第一个样本是尖耳、圆脸、有胡须。

在每个节点上，我们为每个特征计算信息增益，然后在信息增益最高的特征上分裂该节点——方法是比较该节点的熵与分裂后两个节点的加权熵。

所以，根节点包含数据集里的每一个动物。记住 $p_1^{node}$ 是根节点里正类（猫）的比例。所以

$$p_1^{node} = \frac{5}{10} = 0.5$$

现在我们来写一个计算熵的函数。

In [5]:
def entropy(p):
    if p == 0 or p == 1:
        return 0
    else:
        return -p * np.log2(p) - (1- p)*np.log2(1 - p)
    
print(entropy(0.5))

1.0


为演示，我们来计算：如果在每个特征上分裂该节点，各自的信息增益。为此，我们先写几个函数。

In [6]:
def split_indices(X, index_feature):
    """Given a dataset and a index feature, return two lists for the two split nodes, the left node has the animals that have 
    that feature = 1 and the right node those that have the feature = 0 
    index feature = 0 => ear shape
    index feature = 1 => face shape
    index feature = 2 => whiskers
    """
    left_indices = []
    right_indices = []
    for i,x in enumerate(X):
        if x[index_feature] == 1:
            left_indices.append(i)
        else:
            right_indices.append(i)
    return left_indices, right_indices

所以，如果我们选 Ear Shape 来分裂，那么左节点里（对照上表）必须有这些下标：

$$0 \quad 3 \quad 4 \quad 5 \quad 7$$

右节点则是其余的。

In [7]:
split_indices(X_train, 0)

([0, 3, 4, 5, 7], [1, 2, 6, 8, 9])

现在我们需要另一个函数来计算分裂后两个节点的加权熵。如你在视频课里所见，我们必须求出：

- $w^{\text{left}}$ 和 $w^{\text{right}}$，**每个节点** 里动物的比例。
- $p^{\text{left}}$ 和 $p^{\text{right}}$，**每个分支** 里猫的比例。

注意这两个定义的区别！！举例来说，若我们在下标为 0 的特征（Ear Shape）上分裂根节点，那么在含动物 0、3、4、5 和 7 的左节点里，我们有：

$$w^{\text{left}}= \frac{5}{10} = 0.5 \text{ and } p^{\text{left}} = \frac{4}{5}$$
$$w^{\text{right}}= \frac{5}{10} = 0.5 \text{ and } p^{\text{right}} = \frac{1}{5}$$

In [ ]:
def weighted_entropy(X,y,left_indices,right_indices):
    """
    This function takes the splitted dataset, the indices we chose to split and returns the weighted entropy.
    """
    w_left = len(left_indices)/len(X)
    w_right = len(right_indices)/len(X)
    p_left = sum(y[left_indices])/len(left_indices)
    p_right = sum(y[right_indices])/len(right_indices)
    
    weighted_entropy = w_left * entropy(p_left) + w_right * entropy(p_right)
    return weighted_entropy

In [ ]:
left_indices, right_indices = split_indices(X_train, 0)
weighted_entropy(X_train, y_train, left_indices, right_indices)

所以，分裂后两个节点的加权熵是 0.72。要计算 **信息增益**，我们必须用"所选分裂节点（此例是根节点）的熵"减去它。

In [ ]:
def information_gain(X, y, left_indices, right_indices):
    """
    Here, X has the elements in the node and y is theirs respectives classes
    """
    p_node = sum(y)/len(y)
    h_node = entropy(p_node)
    w_entropy = weighted_entropy(X,y,left_indices,right_indices)
    return h_node - w_entropy

In [ ]:
information_gain(X_train, y_train, left_indices, right_indices)

现在，我们来计算：在每个特征上分裂根节点各自的信息增益：

In [ ]:
for i, feature_name in enumerate(['Ear Shape', 'Face Shape', 'Whiskers']):
    left_indices, right_indices = split_indices(X_train, i)
    i_gain = information_gain(X_train, y_train, left_indices, right_indices)
    print(f"Feature: {feature_name}, information gain if we split the root node using this feature: {i_gain:.2f}")
    

所以，最好的分裂特征确实是 Ear Shape。运行下面的代码看看分裂的实际效果。你不需要理解下面这个代码块。

In [ ]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=1, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

这个过程是 **递归的（recursive）**，意味着我们必须对每个节点执行这些计算，直到满足某个停止判据：

- 若分裂后树的深度超过某阈值
- 若得到的节点只有 1 个类别
- 若分裂的信息增益低于某阈值

最终的树看起来是这样：

In [ ]:
tree = []
build_tree_recursive(X_train, y_train, [0,1,2,3,4,5,6,7,8,9], "Root", max_depth=2, current_depth=0, tree = tree)
generate_tree_viz([0,1,2,3,4,5,6,7,8,9], y_train, tree)

恭喜！你完成了这个 notebook！